# Generate Pickle Caches for All Symbols

This notebook generates pickle cache files for all available cryptocurrency symbols in the dataset. This will improve performance for future data loading operations by pre-processing and caching all symbol data.

In [ ]:
import pandas as pd
import pickle
import time
import os
import glob
from pathlib import Path
from datetime import datetime

# Set up paths
DATA_DIR = "/home/craz/crypto/crypto-data/future_data_2"
PICKLE_CACHE_DIR = "/home/craz/crypto/crypto-data/pickle_cache"

# Create pickle cache directory if it doesn't exist
Path(PICKLE_CACHE_DIR).mkdir(parents=True, exist_ok=True)

print(f"Data directory: {DATA_DIR}")
print(f"Pickle cache directory: {PICKLE_CACHE_DIR}")

## 1. Discover Available Symbols

In [ ]:
def get_available_symbols():
    """Get list of all available crypto symbols"""
    return sorted([d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))])

# Get all available symbols
symbols = get_available_symbols()
print(f"Found {len(symbols)} symbols:")
for i, symbol in enumerate(symbols):
    print(f"{i+1:2d}. {symbol}")

## 2. Define Pickle Cache Functions

In [ ]:
def get_pickle_path(symbol):
    """Get the pickle file path for a symbol"""
    return os.path.join(PICKLE_CACHE_DIR, f"{symbol}_full_data.pkl")

def create_pickle_cache(symbol):
    """Create pickle cache for a symbol by loading all CSV data"""
    print(f"\n🔄 Processing {symbol}...")
    start_time = time.time()
    
    symbol_dir = os.path.join(DATA_DIR, symbol)
    if not os.path.exists(symbol_dir):
        print(f"❌ Directory not found: {symbol_dir}")
        return None
    
    # Get all CSV files for the symbol
    csv_files = glob.glob(os.path.join(symbol_dir, "*.csv"))
    csv_files.sort()
    
    if not csv_files:
        print(f"❌ No CSV files found for {symbol}")
        return None
    
    print(f"   Found {len(csv_files)} CSV files")
    
    # Read and combine all data
    dfs = []
    for i, file in enumerate(csv_files):
        try:
            df = pd.read_csv(file)
            dfs.append(df)
            
            # Progress indicator for large datasets
            if len(csv_files) > 50 and (i + 1) % 100 == 0:
                print(f"   Progress: {i+1}/{len(csv_files)} files processed")
                
        except Exception as e:
            print(f"   ⚠️  Error reading {file}: {e}")
            continue
    
    if not dfs:
        print(f"❌ No valid data found for {symbol}")
        return None
    
    # Combine and process data
    print(f"   Combining {len(dfs)} dataframes...")
    combined_df = pd.concat(dfs, ignore_index=True)
    combined_df['open_time'] = pd.to_datetime(combined_df['open_time'])
    combined_df = combined_df.sort_values('open_time')
    
    try:
        # Save data to pickle using pandas method
        pickle_path = get_pickle_path(symbol)
        print(f"   Saving to pickle: {len(combined_df):,} records")
        combined_df.to_pickle(pickle_path)
        
        processing_time = time.time() - start_time
        file_size_mb = os.path.getsize(pickle_path) / (1024 * 1024)
        
        print(f"✅ {symbol}: {len(combined_df):,} records, {file_size_mb:.1f} MB, {processing_time:.2f}s")
        
        return {
            'symbol': symbol,
            'records': len(combined_df),
            'files': len(csv_files),
            'size_mb': file_size_mb,
            'processing_time': processing_time,
            'date_range': {
                'start': combined_df['open_time'].min().isoformat(),
                'end': combined_df['open_time'].max().isoformat()
            }
        }
        
    except Exception as e:
        print(f"❌ Error creating pickle cache for {symbol}: {e}")
        return None

## 3. Generate Pickle Caches for All Symbols

In [ ]:
# Process all symbols
print(f"🚀 Starting pickle cache generation for {len(symbols)} symbols...")
print(f"{'='*80}")

results = []
total_start_time = time.time()
successful = 0
failed = 0

for i, symbol in enumerate(symbols):
    print(f"\n[{i+1}/{len(symbols)}] Processing {symbol}...")
    
    result = create_pickle_cache(symbol)
    if result:
        results.append(result)
        successful += 1
    else:
        failed += 1
    
    # Progress update every 5 symbols
    if (i + 1) % 5 == 0:
        elapsed = time.time() - total_start_time
        avg_time = elapsed / (i + 1)
        remaining = (len(symbols) - i - 1) * avg_time
        print(f"\n📊 Progress: {i+1}/{len(symbols)} ({((i+1)/len(symbols)*100):.1f}%)")
        print(f"   Successful: {successful}, Failed: {failed}")
        print(f"   Estimated time remaining: {remaining/60:.1f} minutes")

total_time = time.time() - total_start_time
print(f"\n{'='*80}")
print(f"🎉 Pickle cache generation completed!")
print(f"   Total time: {total_time/60:.1f} minutes")
print(f"   Successful: {successful}/{len(symbols)} symbols")
print(f"   Failed: {failed}/{len(symbols)} symbols")

## 4. Summary Report

In [ ]:
if results:
    # Create summary DataFrame
    summary_df = pd.DataFrame(results)
    
    print(f"\n📈 Cache Generation Summary:")
    print(f"{'='*80}")
    
    # Overall statistics
    total_records = summary_df['records'].sum()
    total_files = summary_df['files'].sum()
    total_size_mb = summary_df['size_mb'].sum()
    avg_processing_time = summary_df['processing_time'].mean()
    
    print(f"Total records processed: {total_records:,}")
    print(f"Total CSV files processed: {total_files:,}")
    print(f"Total cache size: {total_size_mb:.1f} MB ({total_size_mb/1024:.2f} GB)")
    print(f"Average processing time per symbol: {avg_processing_time:.2f} seconds")
    
    # Top symbols by size
    print(f"\n🔝 Largest datasets:")
    top_by_records = summary_df.nlargest(5, 'records')[['symbol', 'records', 'size_mb', 'processing_time']]
    for _, row in top_by_records.iterrows():
        print(f"   {row['symbol']:12s}: {row['records']:8,} records, {row['size_mb']:6.1f} MB, {row['processing_time']:5.1f}s")
    
    # Date range analysis
    print(f"\n📅 Date range analysis:")
    earliest_start = min(result['date_range']['start'] for result in results)
    latest_end = max(result['date_range']['end'] for result in results)
    print(f"   Data spans from {earliest_start[:10]} to {latest_end[:10]}")
    
    # Display full summary table
    print(f"\n📋 Complete Summary:")
    display(summary_df.sort_values('records', ascending=False))
    
else:
    print("❌ No successful pickle caches were created.")

## 5. Verify Generated Pickle Files

In [ ]:
# Verify all generated pickle files can be loaded
print(f"\n🔍 Verifying generated pickle files...")
print(f"{'='*50}")

verification_results = []
for result in results:
    symbol = result['symbol']
    pickle_path = get_pickle_path(symbol)
    
    try:
        # Test loading with pd.read_pickle
        load_start = time.time()
        df = pd.read_pickle(pickle_path)
        load_time = time.time() - load_start
        
        # Basic validation
        expected_records = result['records']
        actual_records = len(df)
        
        if actual_records == expected_records:
            verification_results.append({
                'symbol': symbol,
                'status': 'OK',
                'records': actual_records,
                'load_time': load_time
            })
            print(f"✅ {symbol:12s}: {actual_records:8,} records loaded in {load_time:.4f}s")
        else:
            verification_results.append({
                'symbol': symbol,
                'status': 'MISMATCH',
                'records': actual_records,
                'expected': expected_records,
                'load_time': load_time
            })
            print(f"⚠️  {symbol:12s}: Record count mismatch! Expected {expected_records:,}, got {actual_records:,}")
            
    except Exception as e:
        verification_results.append({
            'symbol': symbol,
            'status': 'ERROR',
            'error': str(e)
        })
        print(f"❌ {symbol:12s}: Loading failed - {e}")

# Verification summary
successful_verifications = len([r for r in verification_results if r['status'] == 'OK'])
print(f"\n📊 Verification Results:")
print(f"   Successfully verified: {successful_verifications}/{len(verification_results)} files")

if successful_verifications > 0:
    avg_load_time = sum(r['load_time'] for r in verification_results if r['status'] == 'OK') / successful_verifications
    print(f"   Average load time: {avg_load_time:.4f} seconds")

## 6. Cache Directory Status

In [ ]:
# Final cache directory status
print(f"\n📁 Pickle cache directory status:")
print(f"{'='*50}")

cache_files = glob.glob(os.path.join(PICKLE_CACHE_DIR, "*.pkl"))
data_files = [f for f in cache_files if f.endswith('_full_data.pkl')]

print(f"Directory: {PICKLE_CACHE_DIR}")
print(f"Total pickle files: {len(cache_files)}")
print(f"Data files: {len(data_files)}")

if data_files:
    total_cache_size = sum(os.path.getsize(f) for f in data_files) / (1024 * 1024)
    print(f"Total cache size: {total_cache_size:.1f} MB ({total_cache_size/1024:.2f} GB)")

print(f"\n🎯 Pickle cache generation completed successfully!")
print(f"   All symbols are now cached for high-performance loading.")
print(f"   Only full data files saved - no metadata files.")